# 物理仿真与多模态数据生成

本 Demo 分别演示物理执行、轨迹重放渲染、事件流、点云和 Episode 封装 API。


## 0. 初始化平台

只定义工作区与输出目录；场景、任务、方法和仿真参数均在对应 API 单元格中绑定。


In [ ]:
# ruff: noqa
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("未找到 PhysReal-X 项目根目录。")
    PROJECT_ROOT = PROJECT_ROOT.parent
for import_root in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
from examples._common import operation_receipt, output_artifact, write_summary
from physrealx import PhysRealX
from physrealx.errors import PhysRealXError

WORKSPACE = PROJECT_ROOT / ".physrealx/examples/04-simulation-data"
OUTPUT_DIR = PROJECT_ROOT / "examples/simulation_and_data_generation/output"
px = PhysRealX.open(WORKSPACE)
receipts = {}

## 1. 执行物理仿真：`simulations.run`

**用途**：用选定后端执行场景、任务、控制周期和随机种子，生成唯一物理状态轨迹。

**输出**：`StateTrajectory/1`，包含时间、刚体/关节状态、接触和力。


In [ ]:
scene = None  # 替换为已注册 SceneSpec/1 VersionedRef
task = None  # 替换为已注册 TaskSpec/1 VersionedRef
simulation_operation = None
state_trajectory = None
if scene is None or task is None:
    print("[BLOCKED] 请在本单元格绑定场景和任务。")
else:
    try:
        simulation_operation = px.simulations.run(
            scene,
            task,
            method="installed_simulator",
            config={"physics_dt": 0.002, "control_dt": 0.02, "seed": 2026},
        )
        state_trajectory = output_artifact(simulation_operation, "state_trajectory")
        receipts["simulation"] = operation_receipt(simulation_operation)
        print(json.dumps(simulation_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 2. 重放轨迹并渲染：`renders.generate`

**用途**：严格重放上一步状态轨迹，生成 RGB、Depth、Normal、Mask 和 Flow。

**同步约束**：渲染节点不重新运行物理，每帧时间戳和状态摘要必须一致。


In [ ]:
render_operation = None
if state_trajectory is None:
    print("[BLOCKED] 需要先产生状态轨迹。")
else:
    try:
        render_operation = px.renders.generate(
            state_trajectory, method="installed_renderer", config={}
        )
        receipts["rendering"] = operation_receipt(render_operation)
        print(json.dumps(render_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 3. 合成事件相机数据：`events.synthesize`

**用途**：从高帧率强度流生成带时间戳、极性与噪声模型的 Event 流。

**输出**：与 RGB 时钟对齐的 `EventStream/1`。


In [ ]:
rgb_stream = None  # 替换为已验证 RGBStream/1 ArtifactRef
event_operation = None
if rgb_stream is None:
    print("[BLOCKED] 请在本单元格绑定 RGB 强度流。")
else:
    try:
        event_operation = px.events.synthesize(
            rgb_stream,
            method="v2e",
            config={"seed": 2026},
        )
        receipts["event_stream"] = operation_receipt(event_operation)
        print(json.dumps(event_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 4. 反投影点云：`pointclouds.project`

**用途**：用已标定相机模型把米制深度反投影为有组织点云。

**输出**：含坐标系、单位和无效值统计的 `PointCloudStream/1`。


In [ ]:
depth_stream = None  # 替换为已验证 DepthStream/1 ArtifactRef
pointcloud_operation = None
if depth_stream is None:
    print("[BLOCKED] 请在本单元格绑定米制深度流。")
else:
    try:
        pointcloud_operation = px.pointclouds.project(
            depth_stream,
            method="open3d",
            config={},
        )
        receipts["pointcloud_stream"] = operation_receipt(pointcloud_operation)
        print(json.dumps(pointcloud_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 5. 封装 Episode：`episodes.pack`

**用途**：将观测、动作、状态、接触、任务谓词和结果绑定为可回放 Episode。

**发布门禁**：时间线、帧索引、动作和状态完全对齐后才能封装。


In [ ]:
verified_run_evidence = None  # 替换为通过对齐验证的 RunEvidence/1 ArtifactRef
episode_operation = None
if verified_run_evidence is None:
    print("[BLOCKED] 请在本单元格绑定已验证运行证据。")
else:
    try:
        episode_operation = px.episodes.pack(
            verified_run_evidence,
            method="installed_episode_packer",
            config={},
        )
        receipts["episode"] = operation_receipt(episode_operation)
        print(json.dumps(episode_operation.metrics, ensure_ascii=False, indent=2))
    except PhysRealXError as error:
        print(f"[BLOCKED] {error}")

## 6. 保存回执

输出仅收集真实执行的操作和指标。


In [ ]:
report = write_summary(
    OUTPUT_DIR / "simulation-data-report.json",
    {"module": "物理仿真与多模态数据生成", "receipts": receipts},
)
print(json.dumps(report, ensure_ascii=False, indent=2, default=str))